# Query transformation

**Track:** Enterprise Knowledge Assistant · **Stage:** Advanced retrieval

Users rarely ask in the same language as the corpus. Query transformation adds controlled variants: rewriting, multi-query, decomposition, and HyDE-style hypothetical answers. The danger is semantic drift: a transformed query that no longer means what the user asked can retrieve persuasive but wrong evidence.

## What you will build

- A deterministic implementation that runs without API keys.
- A visible trace of evidence, decisions, and failure modes.
- A production design note explaining how this maps to real RAG libraries and systems.

## Concept map

```mermaid
flowchart TD
  Q["Original user question"] --> R["Rewrite"]
  Q --> M["Multi-query variants"]
  Q --> D["Decompose into subquestions"]
  Q --> H["HyDE-style hypothetical answer"]
  R --> U["Union + deduplicate candidates"]
  M --> U
  D --> U
  H --> U
  U --> V["Validate against original intent"]
```

## Setup

Run this notebook from the repository root, or open it in GitHub and copy cells into a local Jupyter session. The helper code lives in `src/enterprise_rag` so the notebook remains readable while the implementation stays testable.

In [ ]:
from pathlib import Path
import sys, json
ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent.parent
sys.path.insert(0, str(ROOT))

def show(obj):
    print(json.dumps(obj, indent=2))

The business query uses “compliance rule” rather than the exact phrase “Regulation R-17.” The variants should recover the vendor and regulation documents without inventing a new compliance story.

In [ ]:
from src.enterprise_rag.lab_experiments import build_enterprise_chunks, query_transformation_report
chunks = build_enterprise_chunks(ROOT / "data/enterprise")
show(query_transformation_report("What compliance rule affects the Atlas database vendor?", chunks))

### Exercise

Add one bad rewrite that changes the meaning of the question. What would you log so a reviewer can see that drift? At minimum: original query, transformed query, retriever route, selected sources, and final answer citations.

## Deliberate failure case

Before moving on, make the system fail on purpose. Change one variable: chunk size, query wording, top-k, reranking terms, route choice, or evaluation labels. Write down whether the failure belongs to ingestion, retrieval, evidence selection, generation, authorization, or operations.

In [ ]:
# Try your own failure experiment here.
# Example: lower top_k to 1, ask an unsupported question, or remove an important query term.
from src.enterprise_rag.lab_experiments import build_enterprise_chunks
question = "What policy covers parental leave?"
chunks = build_enterprise_chunks(ROOT / "data/enterprise")
print("Question:", question)
print("Now change the query, top_k, or chunking strategy and rerun a comparison helper.")

## Reflection questions

1. What did the simplest baseline get right?
2. What failure was invisible until you inspected the trace?
3. Which component would you improve first in production, and how would you prove it helped?
4. What should the system do when evidence is missing, unauthorized, stale, or contradictory?

## References and next reading

- Lewis et al., *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks*: https://arxiv.org/abs/2005.11401
- Stanford IR book: https://nlp.stanford.edu/IR-book/
- LangChain retrieval concepts: https://docs.langchain.com/oss/python/langchain/retrieval
- LlamaIndex RAG guide: https://docs.llamaindex.ai/en/stable/understanding/rag/
- Haystack pipeline docs: https://docs.haystack.deepset.ai/docs/pipelines
- Ragas metrics: https://docs.ragas.io/en/stable/concepts/metrics/